# YEJOO 재고 수불부 STEP 분리 실행 노트북 (V16 기반)

### -*- coding: utf-8 -*-  # 인코딩 설정

YEJOO 재고수불부 자동화 V17 (노트북 → 단일 py 파일)
- 전처리
- STEP0 / STEP0-1
- STEP00 (보세 → 본사 이동 날짜 당김)
- STEP1 ~ STEP5

In [9]:
import pandas as pd  # 데이터 처리
import numpy as np  # 수치 처리
from datetime import timedelta  # 날짜 처리
from datetime import datetime  # 중간저장 타임스탬프

# =========================  # 구분선
# 0) 경로  # 섹션
# =========================  # 구분선
PATH_SUFUL = r"../df_suful_100.xlsx"  # 수불부 경로
PATH_SUBMIT = r"../df_submit_100.xlsx"  # 제출본 경로(미사용)
PATH_BONDED = r"../input data/보세수불부_원본.xlsx"  # 보세수불부 경로
OUT_XLSX = r"대체이력_재고수불부_보정본_최종.xlsx"  # 출력 경로

# =========================  # 구분선
# 1) 컬럼명  # 섹션
# =========================  # 구분선
DATE_COL = "일자"  # 날짜
ITEM_COL = "품목코드"  # 품목코드
NAME_COL = "품목명"  # 품목명
PARTY_COL = "거래처명"  # 거래처명
WH_COL = "창고명"  # 창고명

IN_QTY_COL = "입고수량"  # 입고수량
OUT_QTY_COL = "출고수량"  # 출고수량
STOCK_COL = "재고수량"  # 재고수량

IN_PRICE_COL = "입고단가"  # 입고단가
IN_AMT_COL = "입고금액"  # 입고금액

# =========================  # 구분선
# 2) 옵션  # 섹션
# =========================  # 구분선
STEP00_MAX_PULL_DAYS = 60  # 탐색창(일)
STEP00_WORST_THRESHOLD = -1.0  # 음수 기준(이하만)
VERBOSE = True  # 로그 출력

BONDED_MOVE_TEXT = "[이동] 07보세창고 → 01본사창고"  # 보세→본사 텍스트

STEP00_SAVE_EVERY_N_SUCCESS = 5  # 성공 N건마다 저장
STEP00_SAVE_PREFIX = "중간저장_STEP00"  # 중간저장 접두어

EPS_NEG = -1  # ✅ -0.000... 같은 오차 방지

# =========================  # 구분선
# 3) 유틸  # 섹션
# =========================  # 구분선
def normalize_item_code(x):  # 품목코드 정리 (수불부 "3745" vs 보세 "3745.0" → "3745" 통일, 매칭용)
    if pd.isna(x):  # 결측이면
        return ""  # 빈값
    s = str(x).strip()  # 공백 제거
    try:  # 숫자로 읽히면(보세는 엑셀에서 float로 읽힘) 정수 문자열로 통일
        v = float(s)
        if v == int(v):  # 소수부 없음 (3745.0 등)
            return str(int(v))  # "3745"
        return s  # 3745.5 같은 실제 소수는 그대로
    except (ValueError, TypeError):  # 숫자 아님
        return s  # 문자열 그대로

def safe_date(x):  # 날짜 변환
    return pd.to_datetime(x, errors="coerce", format="mixed")  # 안전 변환

def to_float_series(s):  # float 변환
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype("float64")  # float 통일

def v_or_blank(v):  # 문자열 안전
    if pd.isna(v):  # 결측이면
        return ""  # 빈값
    return str(v)  # 문자열

def str_contains(hay, needle):  # 포함 검사
    if pd.isna(hay):  # 결측이면
        return False  # 아니야
    return needle in str(hay)  # 포함

def same_month(a, b):  # 같은 월인지
    a = pd.Timestamp(a)  # 변환
    b = pd.Timestamp(b)  # 변환
    return (a.year == b.year) and (a.month == b.month)  # 연/월 비교

def qty_equal(a, b, tol=1e-9):  # 수량 일치
    a = float(a)  # float
    b = float(b)  # float
    return abs(a - b) <= tol  # 오차 허용

# =========================  # 구분선
# 4) 로딩 / 전처리  # 섹션
# =========================  # 구분선
def load_excel(path):  # 엑셀 로드
    return pd.read_excel(path)  # 읽기

def preprocess(df):  # 전처리
    df = df.copy()  # 복사
    df[DATE_COL] = safe_date(df[DATE_COL])  # 날짜 변환
    df = df.dropna(subset=[DATE_COL]).copy()  # 날짜 없는 행 제거

    df[ITEM_COL] = df[ITEM_COL].apply(normalize_item_code)  # 품목코드 정리

    if WH_COL not in df.columns:  # 창고명 없으면
        df[WH_COL] = ""  # 생성
    if PARTY_COL not in df.columns:  # 거래처명 없으면
        df[PARTY_COL] = ""  # 생성
    if NAME_COL not in df.columns:  # 품목명 없으면
        df[NAME_COL] = ""  # 생성
    if IN_PRICE_COL not in df.columns:  # 단가 없으면
        df[IN_PRICE_COL] = 0.0  # 생성
    if IN_AMT_COL not in df.columns:  # 금액 없으면
        df[IN_AMT_COL] = 0.0  # 생성

    df[IN_QTY_COL] = to_float_series(df.get(IN_QTY_COL, 0.0))  # 입고 float
    df[OUT_QTY_COL] = to_float_series(df.get(OUT_QTY_COL, 0.0))  # 출고 float
    df[STOCK_COL] = to_float_series(df.get(STOCK_COL, 0.0))  # 재고 float
    df[IN_PRICE_COL] = to_float_series(df.get(IN_PRICE_COL, 0.0))  # 단가 float
    df[IN_AMT_COL] = to_float_series(df.get(IN_AMT_COL, 0.0))  # 금액 float

    return df  # 반환

# =========================  # 구분선
# 5) 재고 재계산(창고+품목)  # 섹션
# =========================  # 구분선
def recalc_inventory(df):  # 재고 재계산 (창고+품목별 누적, 기초재고 반영)
    df = df.copy()  # 복사
    df["_ord"] = np.arange(len(df), dtype="int64")  # 원래순서
    # 이동한 전표(_moved_first=0)는 해당일 맨 앞에 배치 → 같은 날 출고보다 먼저 반영되어 run 방지
    sort_cols = [WH_COL, ITEM_COL, DATE_COL]
    if "_moved_first" in df.columns:
        sort_cols = [WH_COL, ITEM_COL, DATE_COL, "_moved_first", "_ord"]
    else:
        sort_cols = [WH_COL, ITEM_COL, DATE_COL, "_ord"]
    df = df.sort_values(sort_cols)  # 정렬

    out = []  # 결과 모음

    for (wh, code), g in df.groupby([WH_COL, ITEM_COL], sort=False):  # 그룹
        g = g.copy()  # 복사
        # 기초재고: 재고수량이 있는 제일 빠른 일자 행의 재고수량 → 그 다음 행부터 입고/출고 반영
        i0 = None  # 재고수량이 있는 첫 행 인덱스
        for i in range(len(g)):
            v = g.iloc[i][STOCK_COL]
            if pd.notna(v) and str(v).strip() != "":
                i0 = i
                break
        if i0 is None:  # 재고수량 있는 행 없으면 0부터 누적
            run = 0.0
            for i in range(len(g)):
                run += float(g.iloc[i][IN_QTY_COL]) - float(g.iloc[i][OUT_QTY_COL])
                g.iloc[i, g.columns.get_loc(STOCK_COL)] = run
        else:
            run = float(g.iloc[i0][STOCK_COL])  # 기초재고
            g.iloc[i0, g.columns.get_loc(STOCK_COL)] = run  # 해당 행 재고 = 기초
            for i in range(i0 + 1, len(g)):  # 그 다음 행부터
                run += float(g.iloc[i][IN_QTY_COL])  # 입고 +
                run -= float(g.iloc[i][OUT_QTY_COL])  # 출고 -
                g.iloc[i, g.columns.get_loc(STOCK_COL)] = run  # 재고 기록
        out.append(g)  # 누적

    res = pd.concat(out, ignore_index=True) if len(out) > 0 else df.copy()  # 합치기
    if "_ord" in res.columns:  # 보조컬럼 있으면
        res = res.drop(columns=["_ord"])  # 제거
    return res  # 반환

# =========================  # 구분선
# 6) 음수 run 찾기(연속 음수 구간)  # 섹션
# =========================  # 구분선
def find_all_negative_runs(df_item_sorted):  # 음수 run 목록
    stocks = df_item_sorted[STOCK_COL].to_numpy()  # 재고 배열
    dates = df_item_sorted[DATE_COL].to_numpy()  # 날짜 배열

    starts = []  # 시작 인덱스
    in_neg = False  # 음수중 여부
    start_i = None  # 시작점

    for i in range(len(stocks)):  # 순회
        s = float(stocks[i])  # 재고값
        if (s < -EPS_NEG) and (not in_neg):  # ✅ 음수 시작(오차 제외)
            in_neg = True  # 시작 표시
            start_i = i  # 시작 저장
        if (s >= -EPS_NEG) and in_neg:  # ✅ 음수 종료(오차 포함)
            starts.append(int(start_i))  # 저장
            in_neg = False  # 종료
            start_i = None  # 초기화

    if in_neg and (start_i is not None):  # 끝까지 음수면
        starts.append(int(start_i))  # 마지막 저장

    runs = []  # run 목록
    for si in starts:  # 시작점별
        ei = None  # 종료점
        for j in range(si, len(stocks)):  # 이후 탐색
            if float(stocks[j]) >= -EPS_NEG:  # ✅ 복구면
                ei = j  # 종료
                break  # 중단
        if ei is None:  # 복구 못하면
            ei = len(stocks)  # 끝
        worst = float(np.min(stocks[si:ei]))  # 최저 재고
        need_qty = abs(float(stocks[si]))  # 시작 부족량
        runs.append({  # 기록
            "run_start": pd.Timestamp(dates[si]),  # run 시작일
            "start_idx": int(si),  # 시작 idx
            "end_idx": int(ei),  # 종료 idx
            "need_qty": float(need_qty),  # 필요량
            "worst": float(worst),  # 최대 음수
        })  # 추가
    return runs  # 반환

# =========================  # 구분선
# 7) STEP00  # 섹션
#  - ✅ 음수 run 있을 때만 이동  # 주석
#  - ✅ 이동 목표는 "해당 run_start"  # 주석
#  - ✅ 이력이력은 성공만 기록  # 주석
#  - ✅ 보세는 (날짜+품목+수량) 완전일치 후 같이 당김  # 주석
# =========================  # 구분선
def step00_pull_all_wh(df_main, df_bonded):  # STEP00 실행
    df_main = df_main.copy()  # 복사
    df_bonded = df_bonded.copy()  # 복사
    # 이동한 전표는 해당일 맨 앞에 오도록 정렬용 플래그 (0=맨앞, 1=그 외)
    df_main["_moved_first"] = 1
    df_bonded["_moved_first"] = 1

    history_rows = []  # ✅ 성공 이력만
    success_count = 0  # 성공 카운트
    used_main_idx = set()  # 입고 재사용 금지
    failed_run_keys = set()  # 실패 run 스킵

    df_main = recalc_inventory(df_main)  # 재계산
    df_bonded = recalc_inventory(df_bonded)  # 재계산

    keys = df_main[[WH_COL, ITEM_COL]].dropna().drop_duplicates().values.tolist()  # 키 목록
    for wh, code in keys:  # 그룹 반복
        while True:  # run 반복
            g = df_main[(df_main[WH_COL] == wh) & (df_main[ITEM_COL] == code)].copy()  # 그룹 추출
            if len(g) == 0:  # 없으면
                break  # 종료

            # ✅ 음수 자체가 없으면 절대 이동 금지  # 주석
            if float(g[STOCK_COL].min()) >= -EPS_NEG:  # 음수 없음
                break  # 종료

            g = g.sort_values(DATE_COL).reset_index(drop=False)  # 정렬(+원본 index)
            runs = find_all_negative_runs(g)  # run 찾기
            if len(runs) == 0:  # run 없으면
                break  # 종료

            run = None  # 선택 run
            for r in runs:  # 앞 run부터
                rk = (wh, code, pd.Timestamp(r["run_start"]).date())  # 키
                if rk not in failed_run_keys:  # 실패 run 아니면
                    run = r  # 선택
                    break  # 중단
            if run is None:  # 다 실패면
                break  # 종료

            # ✅ 기준 이하 음수만 처리  # 주석
            if float(run["worst"]) > float(STEP00_WORST_THRESHOLD):  # 덜 심각하면
                break  # 종료

            run_start = pd.Timestamp(run["run_start"])  # run 시작일
            end_day = run_start + timedelta(days=int(STEP00_MAX_PULL_DAYS))  # 탐색 끝일

            base_mask = (  # 후보 조건
                (df_main[WH_COL] == wh) &  # 같은 창고
                (df_main[ITEM_COL] == code) &  # 같은 품목
                (df_main[DATE_COL] > run_start) &  # run 이후
                (df_main[DATE_COL] <= end_day) &  # 창 안
                (df_main[IN_QTY_COL] > 0.0)  # 입고만
            )  # 마스크

            cand = df_main[base_mask].copy()  # 후보
            if len(cand) > 0:  # 있으면
                cand = cand[~cand.index.isin(list(used_main_idx))].copy()  # 사용분 제외

            fail_reason = ""  # 실패 사유
            bonded_fail = False  # 보세 실패
            move_type = ""  # 타입
            pick_idx = None  # 선택 idx

            bonded_new_neg = False  # 신규 음수 여부
            bonded_new_neg_min = 0.0  # 신규 최저 재고

            # ---------- 1순위: 보세→본사 이동 ----------  # 주석
            cand_bonded = cand[cand[PARTY_COL].apply(lambda x: str_contains(x, BONDED_MOVE_TEXT))].copy()  # 보세 후보
            if len(cand_bonded) > 0:  # 있으면
                pick_idx = cand_bonded.sort_values(DATE_COL).index[0]  # 1건
                move_type = "BONDED"  # 타입

            # ---------- 2순위: 같은달 매입전표 ----------  # 주석
            if pick_idx is None:  # 보세 없으면
                cand_buy = cand.copy()  # 복사
                cand_buy = cand_buy[~cand_buy[PARTY_COL].astype(str).str.contains(r"\[이동\]", regex=True, na=False)].copy()  # 이동 제외
                cand_buy = cand_buy[(cand_buy[IN_PRICE_COL] > 0.0) & (cand_buy[IN_AMT_COL] > 0.0)].copy()  # 단가/금액
                if len(cand_buy) > 0:  # 있으면
                    cand_buy = cand_buy[cand_buy[DATE_COL].apply(lambda d: same_month(d, run_start))].copy()  # 같은 달
                if len(cand_buy) > 0:  # 남으면
                    pick_idx = cand_buy.sort_values(DATE_COL).index[0]  # 1건
                    move_type = "PURCHASE"  # 타입

            if pick_idx is None:  # 후보 없으면
                fail_reason = "가져올 입고 없음(보세없음+매입없음/조건불충족)"  # 사유

            old_date = ""  # 이동전 날짜
            old_code = ""  # 이동전 코드
            old_name = ""  # 이동전 품목명
            old_party = ""  # 이동전 거래처

            # ✅ 선택이 있으면 값 채우기  # 주석
            if pick_idx is not None:  # 선택됨
                old_date = pd.Timestamp(df_main.loc[pick_idx, DATE_COL])  # 이전 날짜
                old_code = v_or_blank(df_main.loc[pick_idx, ITEM_COL])  # 이전 코드
                old_name = v_or_blank(df_main.loc[pick_idx, NAME_COL])  # 이전 명
                old_party = v_or_blank(df_main.loc[pick_idx, PARTY_COL])  # 이전 거래처

                # ✅ 보세 타입이면 보세도 같이 이동  # 주석
                if move_type == "BONDED":  # 보세면
                    move_qty = float(df_main.loc[pick_idx, IN_QTY_COL])  # 본사 입고수량(이동)

                    before_mask = (df_bonded[ITEM_COL] == code)  # 품목 마스크
                    before_has_neg = bool((df_bonded.loc[before_mask, STOCK_COL] < -EPS_NEG).any()) if bool(before_mask.any()) else False  # 이전 음수
                    before_min = float(df_bonded.loc[before_mask, STOCK_COL].min()) if bool(before_mask.any()) else 0.0  # 이전 최저(참고)

                    mask_bd = (df_bonded[ITEM_COL] == code) & (df_bonded[DATE_COL] == old_date)  # 날짜+품목
                    cand_bd = df_bonded.loc[mask_bd].copy()  # 후보

                    if len(cand_bd) == 0:  # 없으면
                        bonded_fail = True  # 실패
                        fail_reason = "보세 전표 없음(날짜/품목 매칭 실패)"  # 사유
                    else:
                        qty_mask = cand_bd[OUT_QTY_COL].apply(lambda x: qty_equal(x, move_qty))  # 수량 일치
                        cand_bd2 = cand_bd.loc[qty_mask].copy()  # 필터

                        if len(cand_bd2) == 0:  # 없으면
                            bonded_fail = True  # 실패
                            fail_reason = "보세 전표 수량 불일치(날짜/품목은 일치)"  # 사유
                        else:
                            bidx = cand_bd2.index[0]  # 1건 선택
                            df_bonded.loc[bidx, DATE_COL] = run_start  # ✅ 보세 날짜도 당김
                            df_bonded.loc[bidx, "_moved_first"] = 0  # 해당일 맨 앞 배치

                            df_bonded = recalc_inventory(df_bonded)  # ✅ 보세 재계산

                            after_mask = (df_bonded[ITEM_COL] == code)  # 품목 마스크
                            after_has_neg = bool((df_bonded.loc[after_mask, STOCK_COL] < -EPS_NEG).any()) if bool(after_mask.any()) else False  # 이후 음수
                            after_min = float(df_bonded.loc[after_mask, STOCK_COL].min()) if bool(after_mask.any()) else 0.0  # 이후 최저

                            if (not before_has_neg) and after_has_neg:  # ✅ 신규 음수
                                bonded_new_neg = True  # 표시
                                bonded_new_neg_min = after_min  # 기록
                            else:
                                bonded_new_neg = False  # 해제
                                bonded_new_neg_min = 0.0  # 초기화

                # ✅ 실패 없으면 메인도 이동  # 주석
                if fail_reason == "":  # 성공이면
                    df_main.loc[pick_idx, DATE_COL] = run_start  # ✅ 메인 날짜 당김
                    df_main.loc[pick_idx, "_moved_first"] = 0  # 해당일 맨 앞 배치(같은 날 출고보다 먼저 반영)
                    used_main_idx.add(int(pick_idx))  # 재사용 금지

            # ✅ 실패면 이력 기록하지 않음(요구사항)  # 주석
            if fail_reason != "":  # 실패면
                failed_run_keys.add((wh, code, run_start.date()))  # run 포기
                if VERBOSE:  # 로그면
                    print(f"[STEP00] FAIL| {wh} | {code} | run={run_start.date()} | {fail_reason} | type={move_type} | worst={run['worst']}")  # 출력
                continue  # 다음 run

            # ✅ 성공이면 재계산 후 성공 이력만 기록  # 주석
            df_main = recalc_inventory(df_main)  # 메인 재계산
            df_bonded = recalc_inventory(df_bonded)  # 보세 재계산

            new_date = pd.Timestamp(df_main.loc[pick_idx, DATE_COL])  # 이동후 날짜
            new_code = v_or_blank(df_main.loc[pick_idx, ITEM_COL])  # 이동후 코드
            new_name = v_or_blank(df_main.loc[pick_idx, NAME_COL])  # 이동후 명
            new_party = v_or_blank(df_main.loc[pick_idx, PARTY_COL])  # 이동후 거래처

            history_rows.append({  # ✅ 성공 이력만 추가
                "이동전 날짜": old_date.date() if isinstance(old_date, pd.Timestamp) else "",  # 이전 날짜
                "이동전 품목코드": old_code,  # 이전 코드
                "이동전 품목명": old_name,  # 이전 명
                "이동전 거래처명": old_party,  # 이전 거래처
                "해당run의 최대 음수": float(run["worst"]),  # run 최저
                "이동후 날짜": run_start.date() if isinstance(run_start, pd.Timestamp) else "",  # 해당 run_start
                "이동후 품목코드": new_code,  # 이후 코드
                "이동후 품목명": new_name,  # 이후 명
                "이동후 거래처명": new_party,  # 이후 거래처
                "이동타입": move_type,  # 타입
                "run_start": run_start.date(),  # run 시작일
                "창고명": v_or_blank(wh),  # 창고명
                "보세이동후_신규음수여부": True if bonded_new_neg else False,  # 신규 음수
                "보세이동후_최저재고": float(bonded_new_neg_min),  # 최저 재고
            })

            success_count += 1  # 성공 카운트

            if VERBOSE:  # 로그면
                print(f"[STEP00] OK  | {wh} | {code} | {old_date.date()} → {run_start.date()} | type={move_type} | worst={run['worst']}")  # 출력

            if STEP00_SAVE_EVERY_N_SUCCESS > 0 and (success_count % STEP00_SAVE_EVERY_N_SUCCESS == 0):  # 주기 저장
                ts = datetime.now().strftime("%Y%m%d_%H%M%S")  # 시간 문자열
                out_path = f"{STEP00_SAVE_PREFIX}_{ts}.xlsx"  # 파일명
                df_history_mid = pd.DataFrame(history_rows)  # 이력 DF
                with pd.ExcelWriter(out_path, engine="openpyxl") as w:  # 저장
                    df_main.to_excel(w, index=False, sheet_name="수불부_중간")  # 메인
                    df_bonded.to_excel(w, index=False, sheet_name="보세_중간")  # 보세
                    df_history_mid.to_excel(w, index=False, sheet_name="이력이력_중간")  # 이력
                print("중간저장 완료:", out_path)  # 로그

    df_history = pd.DataFrame(history_rows)  # 최종 이력 DF
    if "_moved_first" in df_main.columns:
        df_main = df_main.drop(columns=["_moved_first"])
    if "_moved_first" in df_bonded.columns:
        df_bonded = df_bonded.drop(columns=["_moved_first"])
    return df_main, df_bonded, df_history  # 반환

# =========================  # 구분선
# 8) 메인  # 섹션
# =========================  # 구분선
def main():  # 메인
    df_main = preprocess(load_excel(PATH_SUFUL))  # 수불부 로드
    _df_submit = load_excel(PATH_SUBMIT)  # 제출본 로드(미사용)
    df_bonded = preprocess(load_excel(PATH_BONDED))  # 보세 로드

    df_main, df_bonded, df_history = step00_pull_all_wh(df_main, df_bonded)  # STEP00 실행

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:  # 저장
        df_main.to_excel(w, index=False, sheet_name="수불부_보정본")  # 메인 저장
        df_bonded.to_excel(w, index=False, sheet_name="보세수불부_보정본")  # 보세 저장
        df_history.to_excel(w, index=False, sheet_name="이력이력")  # 이력 저장

    print("완료:", OUT_XLSX)  # 완료 로그

if __name__ == "__main__":  # 진입점
    
    main()  # 실행


[STEP00] FAIL| 01본사창고 | 1100 | run=2025-05-08 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-34.0
[STEP00] OK  | 01본사창고 | 1163 | 2025-04-02 → 2025-03-12 | type=BONDED | worst=-13.900000000000091
[STEP00] OK  | 01본사창고 | 2518 | 2025-05-21 → 2025-05-16 | type=BONDED | worst=-1.0
[STEP00] FAIL| 01본사창고 | 2518 | run=2025-07-03 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-1.0
[STEP00] FAIL| 01본사창고 | 2518 | run=2025-11-10 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-51.0
[STEP00] FAIL| 01본사창고 | 2583 | run=2025-09-11 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-1.0
[STEP00] FAIL| 01본사창고 | 2727 | run=2025-01-03 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-1.0
[STEP00] FAIL| 01본사창고 | 2727 | run=2025-01-07 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-1.0
[STEP00] FAIL| 01본사창고 | 2727 | run=2025-01-08 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-32.0
[STEP00] FAIL| 01본사창고 | 2727 | run=2025-02-12 | 가져올 입고 없음(보세없음+매입없음/조건불충족) | type= | worst=-11.0
[STEP00] FAIL| 01본사창고 | 2727 | run=2025-02-13 | 가

KeyboardInterrupt: 